# Chapter 7: Advanced Agents and Tools

In this chapter's notebook, we demonstrate advanced techniques for developing and deploying agent and tool-integrated agents by using MLflow's `ResponsesAgent` interface and Custom Tools. In this chapter, you will:

- Build and customize advanced `ResponsesAgent` models with MLflow
- Integrate external tools—like SQL functions, API calls, and custom data lookups—into tool-calling agent workflows
- Implement streaming responses, retain conversation history, and orchestrate multi-agents supervisor
- Package, version, and deploy generative AI agent applications using custom PyFuncs and MLflow Model Serving

## 📓 About this notebook
This notebook builds an advanced Unity Airways booking agent: creating Unity Catalog SQL lookup tools, a vector retriever tool, an API-calling tool, and an MCP-backed tool, then packaging the LangGraph agent with MLflow's ResponsesAgent interface for evaluation, logging, and Unity Catalog registration.

**Maps to the book:** Chapter 7, *Advanced Agents and Tools* sections: Developing Agents with MLflow (MLflow AI Gateway Interface); Creating Tools (Vector Retriever Tool, Structured Data Lookup Tool, API-Calling Tool); Packaging and Deploying with ResponsesAgent; Integrating Advanced Capabilities (Context Engineering, MCP Servers as Tools, Multiagents Orchestration).

### ✅ Prerequisites

Run these before this notebook:

1. [`Appendix/data_ingestion`](../Appendix/data_ingestion): creates the booking/FAQ tables and the FAQ **vector search index** used by the retriever tool.

You also need permission to **create Unity Catalog functions** (the SQL tools) and query a Vector Search endpoint. See the [repository README](../README.md).

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

## Agent tools
First, let's develop the tools that we will need. 

We'll need:
1. SQL Tool to retrieve customer bookings from Unity Airways booking data table. > structured data retrieval
2. Vector Retriever tool for retrieving relevant question / answer from Unity Airways FAQ. > unstructured data retrieval

For more examples of tools to add to your agent, see Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/agent-tool) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/agent-tool))


### Structured data lookup tool
In this section, we create Unity Catalog SQL functions the agent can call as tools to look up and modify Unity Airways booking records _(see Ch 7, "Structured Data Lookup Tool")_.

#### 1. SQL Tool for Customer Booking Retrieval

In [0]:
%sql
CREATE OR REPLACE FUNCTION workspace.unity_airways.lookup_customer_info(
  user_email STRING COMMENT 'Email of the customer whose info to look up.'
)
RETURNS TABLE
COMMENT 'Returns all metadata about a specific customer as a markdown table.'
RETURN (
  SELECT *
  FROM workspace.unity_airways.booking_records_dataset
  WHERE `primary_contact_email` LIKE user_email

)

##### Test the function
Always a good practice to test your function and check it works as expected. Specify a fully qualified function name in the execute_function API to run the function:

In [0]:
import pandas as pd 
from io import StringIO
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.lookup_customer_info",
  parameters={"user_email": 'alex.lim@example.com'}
)

pd.read_csv(StringIO(result.value))

In [0]:
from databricks_langchain import UCFunctionToolkit
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

# Create the toolkit with the function
toolkit = UCFunctionToolkit(function_names=["workspace.unity_airways.lookup_customer_info"], client=client)

# Retrieve the LangChain tool object
tool = toolkit.tools[0]

# Directly test (invoke) the tool
result = tool.invoke({"user_email": "alex.lim@example.com"})
result

In [0]:
import pandas as pd
import json
from io import StringIO


# Load as JSON and extract the CSV
data = json.loads(result)["value"]

# Use StringIO to treat the string as a file for pandas
df = pd.read_csv(StringIO(data))

# Show the DataFrame
display(df)


#### 2. SQL Tool to Modify or Cancel Booking



In [0]:
%sql
CREATE OR REPLACE FUNCTION workspace.unity_airways.modify_cancel_booking(
  user_booking_id STRING COMMENT 'ID of the flight to change or cancel.',
  intent STRING COMMENT 'Intent of action: "modify" or "cancel"'
)
RETURNS TABLE(
  booking_id STRING,
  decision STRING COMMENT 'approved | denied | needs_agent', 
  status STRING,
  coupon_status STRING,
  refunded_amount DOUBLE,
  reason STRING COMMENT 'Explanation for the decision'
)
COMMENT 'Changes date/time or cancels booking, and gives back what decision is made: Decisions: approved | denied | needs_agent'
RETURN (
  WITH booking_info AS (
    SELECT 
      booking_id,
      status,
      policy_refundable,
      policy_changeable, 
      total_amount,
      refunded_amount,
      waiver_code,
      irrops_flag,
      schedule_change_flag,
      DATEDIFF(DAY, CURRENT_DATE(), travel_start_date) as days_until_travel
    FROM workspace.unity_airways.booking_records_dataset
    WHERE booking_id = user_booking_id
  ),
  
  decision_result AS (
    SELECT 
      bi.*,
      CASE 
        -- Deny invalid requests
        WHEN bi.booking_id IS NULL THEN 'denied'
        WHEN bi.status IN ('CANCELLED', 'REFUNDED') THEN 'denied' 
        WHEN intent NOT IN ('modify', 'cancel') THEN 'denied'

        -- Modifications always need agent attention  
        WHEN intent = 'modify' THEN 'needs_agent'
        
        -- High value cancellations need agent approval (>$500)
        WHEN intent = 'cancel' AND bi.total_amount > 500 THEN 'needs_agent'
        
        -- Auto approve simple cancellations
        ELSE 'approved'
      END as decision,
      
      CASE 
        -- Denial reasons
        WHEN bi.booking_id IS NULL THEN 'Booking ID not found'
        WHEN bi.status IN ('CANCELLED', 'REFUNDED') THEN 'Booking already processed'
        WHEN intent NOT IN ('modify', 'cancel') THEN 'Invalid intent - must be "modify" or "cancel"'
        WHEN intent = 'modify' AND bi.policy_changeable = FALSE AND bi.waiver_code IS NULL THEN 'Ticket is non-changeable'
        WHEN intent = 'cancel' AND bi.policy_refundable = FALSE AND bi.waiver_code IS NULL AND bi.irrops_flag = FALSE THEN 'Ticket is non-refundable'
        
        -- Approval reasons
        WHEN bi.waiver_code IS NOT NULL THEN 'Approved with waiver code'
        WHEN bi.irrops_flag = TRUE THEN 'Approved due to airline operational disruption'
        WHEN bi.schedule_change_flag = TRUE THEN 'Approved due to schedule change'
        
        -- Agent review reasons
        WHEN intent = 'modify' THEN 'Modification requires agent review'
        WHEN intent = 'cancel' AND bi.total_amount > 500 THEN 'High-value cancellation requires agent approval'
        
        -- Default approval
        ELSE 'Standard cancellation approved'
      END as reason,
      
      CASE 
        WHEN intent = 'cancel' THEN
          CASE
            WHEN bi.waiver_code IS NOT NULL OR bi.irrops_flag = TRUE OR bi.schedule_change_flag = TRUE 
            THEN bi.total_amount  -- Full refund for waivers
            WHEN bi.policy_refundable = TRUE 
            THEN bi.total_amount * 0.8  -- 80% refund for refundable tickets
            ELSE 0.0  -- No refund for non-refundable
          END
        ELSE 0.0  -- No refund for modifications
      END as calculated_refund
    FROM booking_info bi
  )
  
  SELECT 
    dr.booking_id,
    dr.decision,
    CASE 
      WHEN dr.decision = 'approved' AND intent = 'cancel' THEN 'CANCELLED'
      WHEN dr.decision = 'approved' AND intent = 'modify' THEN 'MODIFIED'
      WHEN dr.decision = 'needs_agent' THEN 'PENDING_APPROVAL'
      ELSE dr.status
    END as status,
    CASE 
      WHEN dr.decision = 'approved' AND intent = 'cancel' THEN 'REFUNDED'
      WHEN dr.decision = 'approved' AND intent = 'modify' THEN 'EXCHANGED' 
      WHEN dr.decision = 'needs_agent' THEN 'PENDING'
      ELSE 'ORIGINAL'
    END as coupon_status,
    CASE 
      WHEN dr.decision = 'approved' AND intent = 'cancel' THEN dr.calculated_refund
      ELSE COALESCE(dr.refunded_amount, 0.0)
    END as refunded_amount,
    dr.reason
  FROM decision_result dr
);


### Vector retriever tool
Wrap a Databricks Vector Search index as a retriever tool so the agent can answer FAQ questions from unstructured policy documents. _(see Ch 7, "Vector Retriever Tool")_

In [0]:
import pandas as pd 
from io import StringIO
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

print("Test 1: This ticket is already refunded by the airline.")
client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.modify_cancel_booking",
  parameters={"user_booking_id": "xxc02dgtfgq5c34d",
              "intent": "cancel"}
)
display(pd.read_csv(StringIO(result.value)))

print("Test 2: This ticket fare is higher than 1000 and should be handled by an agent.")
client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.modify_cancel_booking",
  parameters={"user_booking_id": "cgvyie6ivwpvs7hz",
              "intent": "cancel"}
)
display(pd.read_csv(StringIO(result.value)))

print("Test 3: This ticket should be approved for refund.")
client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.modify_cancel_booking",
  parameters={"user_booking_id": "c3dd03uvulemx15z",
              "intent": "cancel"}
)
display(pd.read_csv(StringIO(result.value)))

#### 3. Vector Retriever Tool for FAQ Retrieval

In [0]:
from databricks_langchain import VectorSearchRetrieverTool

# Use Databricks vector search index as tool
vector_retriever_tool = VectorSearchRetrieverTool(
                          index_name="workspace.unity_airways.faq_index",
                          num_results=5,
                          tool_description="Search through unity airways frequently asked question (FAQ) about flight cancellation, travel policies and baggages security."
                          )

##### Test the tool
Test the call to your retrieval tool. Specify a question and check if the results returned are coherent to the setting set on the retriever above.

In [0]:
results = vector_retriever_tool.invoke({
    "query": "Can my battery pack be transported in cabin baggage?"
})


So far, so good. Our two tools are working as expected.

In [0]:
results = vector_retriever_tool.invoke({
    "query": "What are the amenities in the plane?"
})

### API-calling tool
Another type of useful implementation is to expose an external weather API as a structured tool, letting the agent fetch live forecasts during a conversation. _(see Ch 7, "API-Calling Tool")_

### 4. Weather Forecast Tool using API

In [0]:
from typing import Dict, Any
import openmeteo_requests
import pandas as pd
from pydantic import BaseModel
from langchain_core.tools import StructuredTool

class Localization(BaseModel):
    latitude: float
    longitude: float

def get_weather_forecast(latitude: float, longitude: float) -> Dict[str, Any]:
    """
    Tool to fetch hourly weather forecast from Open-Meteo.
    Returns dict with summary and dataframe (truncated for brevity), suitable for LangGraph tool node.
    """
    openmeteo = openmeteo_requests.Client()
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude, # 1.3667 Singapore's latitude
        "longitude":  longitude, # 103.8 Singapore's longitude
        "hourly": ["temperature_2m", "rain", "precipitation_probability"],
    }
    responses = openmeteo.weather_api(url, params=params)    
    hourly = responses[0].Hourly()
    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "rain": hourly.Variables(1).ValuesAsNumpy(),
        "precipitation_probability": hourly.Variables(2).ValuesAsNumpy(),
    }    
    return hourly_data
    

weather_tool = StructuredTool(
    name="get_weather_forecast",
    func=get_weather_forecast,
    description="Get weather forecast given latitude and longitude.",
    args_schema=Localization,
)

In [0]:
# Call the tool by passing an instance of the input schema
result = weather_tool.invoke({"latitude": 1.3667, "longitude": 103.8})
display(pd.DataFrame(result))

### 5. MCP Server
MCP servers are also a popular integration to enhance agents. Let's connect to a managed Databricks MCP server and adapt its tools into LangChain-compatible tools for the agent. _(see Ch 7, "MCP Servers As Tools")_

### Packaging the agent with ResponsesAgent
We can wrap the LangGraph tool-calling agent in MLflow's ResponsesAgent interface to support multi-turn conversations and streaming, then write it to agent.py for logging. _(see Ch 7, "Packaging the Agent with MLflow ResponsesAgent")_

In [0]:
%%writefile agent.py
from typing import Annotated, Any, Generator, List, Optional, Sequence, TypedDict, Union

import mlflow
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient, DatabricksOAuthClientProvider
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.prebuilt.tool_node import ToolNode
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client as connect
from mlflow.pyfunc import ResponsesAgent
from pydantic import create_model

workspace_client = WorkspaceClient()
host = workspace_client.config.host
MANAGED_MCP_SERVER_URL = f"{host}/api/2.0/mcp/vector-search/workspace/unity_airways"

# Define a custom LangChain tool that wraps functionality for calling MCP servers
class MCPTool(BaseTool):
    """Custom LangChain tool that wraps MCP server functionality"""
    def __init__(
        self,
        name: str,
        description: str,
        args_schema: type,
        server_url: str,
        ws: WorkspaceClient,
    ):
        super().__init__(name=name, description=description, args_schema=args_schema)
        object.__setattr__(self, "server_url", server_url)
        object.__setattr__(self, "workspace_client", ws)

    def _run(self, **kwargs) -> str:
        """Execute the MCP tool"""
        # Use managed MCP server via synchronous call
        mcp_client = DatabricksMCPClient(
            server_url=self.server_url, workspace_client=self.workspace_client
        )
        response = mcp_client.call_tool(self.name, kwargs)
        return "".join([c.text for c in response.content])


# Convert an MCP tool definition into a LangChain-compatible tool
def create_langchain_tool_from_mcp(
    mcp_tool, server_url: str, ws: WorkspaceClient
):
    """Create a LangChain tool from an MCP tool definition"""
    schema = mcp_tool.inputSchema.copy()
    properties = schema.get("properties", {})
    required = schema.get("required", [])

    # Map JSON schema types to Python types for input validation
    TYPE_MAPPING = {"integer": int, "number": float, "boolean": bool}
    field_definitions = {}
    for field_name, field_info in properties.items():
        field_type_str = field_info.get("type", "string")
        field_type = TYPE_MAPPING.get(field_type_str, str)

        if field_name in required:
            field_definitions[field_name] = (field_type, ...)
        else:
            field_definitions[field_name] = (field_type, None)

    # Dynamically create a Pydantic schema for the tool's input arguments
    args_schema = create_model(f"{mcp_tool.name}Args", **field_definitions)
    print("args_schema: ", args_schema)
    # Return a configured MCPTool instance
    return MCPTool(
        name=mcp_tool.name,
        description=mcp_tool.description or f"Tool: {mcp_tool.name}",
        args_schema=args_schema,
        server_url=server_url,
        ws=ws
        )


mcp_tool = DatabricksMCPClient(server_url=MANAGED_MCP_SERVER_URL, workspace_client=workspace_client).list_tools()
tool = create_langchain_tool_from_mcp(mcp_tool[0], MANAGED_MCP_SERVER_URL, workspace_client)

In [0]:
from agent import tool
tool.run(tool_input={"query": "What is the airline's mission"})


## Define the agent in code
Below, we define the agent code in a single cell. This lets you easily write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.




### 1. Wrap the LangGraph agent using the `ResponsesAgent` interface

For compatibility with Databricks AI features, the `LangGraphResponsesAgent` class implements the `ResponsesAgent` interface to wrap the LangGraph agent.

Databricks recommends using `ResponsesAgent` as it simplifies authoring multi-turn conversational agents using an open source standard. See MLflow's [ResponsesAgent documentation](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/).


In [0]:
!pip list | grep openmeteo_requests

In [0]:
%%writefile agent.py
import json
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict, Union, Dict
from uuid import uuid4
import openmeteo_requests
import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    VectorSearchRetrieverTool,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    BaseMessage,
    convert_to_openai_messages,
)
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, StructuredTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from langgraph.prebuilt import ToolNode
import pandas as pd
from pydantic import BaseModel

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = """You are an helpful flight assistant for Unity Airways, capable of supporting users with their bookings and policy questions using tools to retrieve information from Unity Airways knowledge bases.

Available user requests you can handle:
	•	Help customers find their flight reservations using booking reference (PNR), last name with email or phone, or ticket number.
	•	Assist in changing flight dates/times or canceling bookings, calculating and informing users about any applicable fees or waivers, and processing cancellations or changes.
	•	Answer customer questions about baggage allowances, change and refund rules, or disruption waivers by searching the policy FAQ.

You have access to these tools:
	•	lookup_booking: Given a PNR, ticket number, or name plus email/phone, retrieve full booking and policy context.
	•	modify_cancel_booking: Given booking context, change or cancel bookings, determine fees/waivers, and update booking state.
	•	faq_search: Given a natural language policy question, return the best answer, its source, and a confidence score.

When a customer asks a question:
	1.	Identify the primary intent: find booking, modify/cancel, or a policy query.
	2.	Gather necessary information (prompt the customer for any missing details).
	3.	Use the correct tool(s) to answer the request or perform the action.
	4.	For booking or change/cancel actions, clearly explain the outcome, including eligibility, rules, fees, refunds, or waivers.
	5.	If a policy query is made, search the FAQ and provide the most relevant, accurate information with attribution.
	6.	Escalate to a human agent if needed.
 
Always remain friendly, concise, and clear. Always confirm required information before taking any action. Make sure your responses are personalized and relevant to the user's intent. If the action asked cannot be handled, politely say so, and redirect to customer service contact email: support@unityairways.com. """

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## To create and see usage examples of more tools, see https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

# You can use UDFs in Unity Catalog as agent tools
UC_TOOL_NAMES = ["workspace.unity_airways.lookup_customer_info", "workspace.unity_airways.modify_cancel_booking"]
uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
tools.extend(uc_toolkit.tools)

#############################
## Vector Search for FAQ Tool
#############################
# Use Databricks vector search indexes as tools
VECTOR_SEARCH_TOOLS = []

VECTOR_SEARCH_TOOLS.append(
    VectorSearchRetrieverTool(
        index_name="workspace.unity_airways.faq_index",
        num_results=5,
        tool_description="Search through unity airways frequently asked question (FAQ) about flight cancellation, travel policies and baggages security."
    )
)
tools.extend(VECTOR_SEARCH_TOOLS)
#####################
## Define agent logic
#####################

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
):
    print(tools)
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there are function calls, continue. else, end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        else:
            return "end"

    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(AgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ToolNode(tools))
    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        self.agent = agent

    def _responses_to_cc(self, message: dict[str, Any]) -> list[dict[str, Any]]:
        """Convert from a Responses API output item to ChatCompletion messages."""
        msg_type = message.get("type")
        if msg_type == "function_call":
            return [
                {
                    "role": "assistant",
                    "content": "tool call",
                    "tool_calls": [
                        {
                            "id": message["call_id"],
                            "type": "function",
                            "function": {
                                "arguments": message["arguments"],
                                "name": message["name"],
                            },
                        }
                    ],
                }
            ]
        elif msg_type == "message" and isinstance(message["content"], list):
            return [
                {"role": message["role"], "content": content["text"]}
                for content in message["content"]
            ]
        elif msg_type == "reasoning":
            return [{"role": "assistant", "content": json.dumps(message["summary"])}]
        elif msg_type == "function_call_output":
            return [
                {
                    "role": "tool",
                    "content": message["output"],
                    "tool_call_id": message["call_id"],
                }
            ]
        compatible_keys = ["role", "content", "name", "tool_calls", "tool_call_id"]
        filtered = {k: v for k, v in message.items() if k in compatible_keys}
        return [filtered] if filtered else []

    def _prep_msgs_for_cc_llm(self, responses_input) -> list[dict[str, Any]]:
        "Convert from Responses input items to ChatCompletion dictionaries"
        cc_msgs = []
        for msg in responses_input:
            cc_msgs.extend(self._responses_to_cc(msg.model_dump()))

    def _langchain_to_responses(self, messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
        "Convert from ChatCompletion dict to Responses output item dictionaries"
        for message in messages:
            message = message.model_dump()
            role = message["type"]
            if role == "ai":
                if tool_calls := message.get("tool_calls"):
                    return [
                        self.create_function_call_item(
                            id=message.get("id") or str(uuid4()),
                            call_id=tool_call["id"],
                            name=tool_call["name"],
                            arguments=json.dumps(tool_call["args"]),
                        )
                        for tool_call in tool_calls
                    ]
                else:
                    return [
                        self.create_text_output_item(
                            text=message["content"],
                            id=message.get("id") or str(uuid4()),
                        )
                    ]
            elif role == "tool":
                return [
                    self.create_function_call_output_item(
                        call_id=message["tool_call_id"],
                        output=message["content"],
                    )
                ]
            elif role == "user":
                return [message]

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = []
        for msg in request.input:
            cc_msgs.extend(self._responses_to_cc(msg.model_dump()))

        for event in self.agent.stream({"messages": cc_msgs}, stream_mode=["updates", "messages"]):
            if event[0] == "updates":
                for node_data in event[1].values():
                    for item in self._langchain_to_responses(node_data["messages"]):
                        yield ResponsesAgentStreamEvent(type="response.output_item.done", item=item)
            # filter the streamed messages to just the generated text messages
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except Exception as e:
                    print(e)


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
mlflow.langchain.autolog()
agent = create_tool_calling_agent(llm, tools, system_prompt)
response_agent = LangGraphResponsesAgent(agent)
mlflow.models.set_model(response_agent)

In [0]:
from agent import agent
result = agent.invoke({"messages": [{"role": "user", "content": "can the booking c3dd03uvulemx15z be cancelled?"}]})
result

In [0]:
from IPython.display import Image, display

# Comment in the agent code `%%writefile agent.py` if you want to run the graph visualization
display(Image(agent.get_graph().draw_mermaid_png()))

### 2. Test the agent

Let's test the agent by interacting with it. Since this notebook called `mlflow.langchain.autolog()`, you can view the trace for each step the agent takes.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
from agent import response_agent
result = response_agent.predict({"input": [{"role": "user", "content": "can the booking c3dd03uvulemx15z be cancelled?"}]})
result

### Smoke test: verify agent responds
Testing the agent to verify it can process requests and call tools.

In [0]:
result = response_agent.predict({"input": [{"role": "user", "content": "what is the weather in singapore for the upcoming week?"}]})
result

### Validating and registering the agent
You can run response streaming validation with mlflow.models.predict_stream to check if everything works properly, then register the packaged ResponsesAgent to Unity Catalog so it is ready for Model Serving. _(see Ch 7, "Register the ResponsesAgent with MLflow in Unity Catalog")_

In [0]:
for chunk in response_agent.predict_stream({"input": [{"role": "user", "content": "what's the policy for rescheduling my flight?"}]}):
    print(chunk.model_dump(exclude_none=True))

### 3. Log the agent as an MLflow model

Now we can log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`

  - **Note:** If your Unity Catalog tool queries a vector search index or leverages external functions, you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).



In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
from agent import UC_TOOL_NAMES, VECTOR_SEARCH_TOOLS
import mlflow
from mlflow.models.resources import DatabricksFunction
from pkg_resources import get_distribution

resources = []
for tool in VECTOR_SEARCH_TOOLS:
    resources.extend(tool.resources)
for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            "databricks-langchain",
            f"langgraph=={get_distribution('langgraph').version}",
            f"backoff=={get_distribution('backoff').version}",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"openmeteo_requests=={get_distribution('openmeteo_requests').version}",
        ],
        resources=resources,
    )

## Evaluate the tools with Agent Evaluation

Use Mosaic AI Agent Evaluation to evaluate the agent's responses based on expected responses and other evaluation criteria. The evaluation criteria you specify will guide iterations, and MLflow will help track the computed quality metrics.
See Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).


To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Can I bring my battery to cabin luggage?"}]},
        "expected_response": "Spare lithium batteries and power banks can be in carry-on only with terminals protected. Size limits apply per IATA guidance.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: response_agent.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## Prepare for Deployment


### 1. Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API with the environment manager specified. This sets up a virtual environment and ensures that all the libraries are specified in the requirements and testing the prediction against it. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Can my battery pack be transported in cabin baggages in this weather in singapore?"}]},
    env_manager="uv",
)

### 2. Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog. After that, we will look at how to deploy the agent in the next chapter.


In [0]:
mlflow.set_registry_uri("databricks-uc")

catalog = "workspace"
schema = "unity_airways"
model_name = "unity-airways-booking-agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)